# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² tabular dataset of cancer survivors with second primary colorectal cancer using the `mlcroissant` library and pandas. All references to data entities (record sets, fields, and columns) use their unique Croissant `@id` identifiers to ensure schema consistency.

### Dataset Source
The dataset is described by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and available records using `mlcroissant`. This fetches the Croissant schema and prepares for record set exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}\n{metadata.description}\n")
print(f"Croissant Dataset @id: {metadata.id}")

## 2. Data Overview
Let's enumerate the available record sets, their `@id` values, and for each, list their corresponding fields and columns by `@id`.
This allows precise selection of entities for subsequent steps.

In [ ]:
# Retrieve and display all record set @ids and fields
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs.id}  --  name: {getattr(rs, 'name', '')}")
        # List all field @ids in this record set
        for f in rs.fields:
            print(f"  Field: {f.id:60} name: {getattr(f, 'name', '')}")
            if hasattr(f, 'columns') and f.columns:
                for c in f.columns:
                    print(f"    Column: {c.id:60} name: {getattr(c, 'name', '')}")
        print("-")

## 3. Data Extraction
For each available record set, extract its records to a pandas DataFrame using the record set's `@id`.
The record set and field `@id`s identified above are used in the code below.

**Note:** Depending on the dataset, you might have a single or multiple record sets to extract. All extractions use `@id` to specify which to load.

In [ ]:
# Gather all available record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for RecordSet @id: {rs_id} -- shape: {dataframes[rs_id].shape}")

# Example: print the first few columns and rows of the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns for {first_rs_id}:\n", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process selected numeric and categorical fields for initial exploration.
- **Filtering:** We'll filter rows based on a numeric field's threshold.
- **Normalization:** We'll standardize a numeric field.
- **Grouping:** We'll group and aggregate by a categorical/grouping field if available.

All column/field references use the `@id` string for clarity and reproducibility.

In [ ]:
# --- Identify a numeric field and a group field for demonstration --- #
# Inspect the columns printed above and select appropriate @ids.
# As an example, let's assume the following @ids are present: 
# (Replace with the actual @ids from your dataset overview. Adjust below variables as needed.)

# Example field/column @ids (you MUST replace these with actual @ids from the data overview step):
numeric_field_id = None
group_field_id = None
first_rs_id = record_set_ids[0] if record_set_ids else None

if first_rs_id:
    df = dataframes[first_rs_id]
    numeric_candidates = df.select_dtypes(include=['number']).columns
    if numeric_candidates.any():
        numeric_field_id = numeric_candidates[0]  # use first numeric field as example
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric fields found.")
        numeric_field_id = None
    # Pick a non-numeric field as group if available
    non_numeric_candidates = df.select_dtypes(exclude=['number']).columns
    if non_numeric_candidates.any():
        group_field_id = non_numeric_candidates[0]
        print(f"Using group/categorical field: {group_field_id}")
    else:
        print("No categorical/group fields found.")
        group_field_id = None

    # If we found a numeric field, do filtering/normalizing/grouping
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f} (mean)")
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
        )
        print(f"Top rows with normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group and aggregate (mean) if group field exists
        if group_field_id and group_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Group mean of {numeric_field_id} by {group_field_id}:")
            display(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and optionally the mean value per group using matplotlib/seaborn.
All visualization uses `@id` for axis labels.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_rs_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    # Histogram of the numeric field
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # Grouped mean bar plot if available
    if group_field_id and group_field_id in df.columns:
        mean_by_group = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=mean_by_group)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=60)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored a Croissant FAIR² clinical oncology dataset using `mlcroissant`, listing all record sets, fields, and columns by their `@id`. We extracted records into pandas DataFrames, filtered and normalized fields, grouped by key attributes, and visualized data distributions. This workflow provides a template for systematic and reproducible data exploration using Croissant metadata references.
